In [49]:
import requests
from bs4 import BeautifulSoup
import pandas as pd

import time


In [27]:
urls = {
    "Charles Oliveira": "http://ufcstats.com/fighter-details/07225ba28ae309b6",
    "Max Holloway": "http://ufcstats.com/fighter-details/150ff4cc642270b9"
}

In [28]:
def get_fighter_html(url):
    headers = {
         'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36'
    }
    response = requests.get(url, headers=headers)

    if response.status_code == 200:
        return BeautifulSoup(response.content, 'html.parser')
    else:
        print(f"deu erro {response.status_code}")
        return None

In [29]:
charles = get_fighter_html(urls["Charles Oliveira"])
print("HTML do Charles baixado com sucesso! Tamanho:", len(str(charles)))

HTML do Charles baixado com sucesso! Tamanho: 109424


## Caracteristicas dos lutadores

In [30]:
def extract_fighter_bio(soup, name):
    fighter_data = {"name": name}

    info_box = soup.find('div', class_='b-list__info-box_style_small-width')

    if info_box:
        items = info_box.find_all('li')
        for item in items:
            text_line = item.text.strip().split('\n')
            if len(text_line) >= 2:
                key = text_line[0].strip().replace(':', '')
                value = text_line[-1].strip()
                fighter_data[key] = value
    
    return fighter_data

In [31]:
charles_bio = extract_fighter_bio(charles, "Charles Oliveira")
print("dados do Charles:")
for k,v in charles_bio.items():
    print(f"{k}: {v}")

dados do Charles:
name: Charles Oliveira
Height: 5' 10"
Weight: 155 lbs.
Reach: 74"
STANCE: Orthodox
DOB: Oct 17, 1989


### Puxando estatisticas completas 

In [32]:
def extract_fighter_stats(soup, name):
    fighter_data = {"name": name}

    info_lists = soup.find_all('ul', class_='b-list__box-list')

    if not info_lists:
        return fighter_data

    for info_list in info_lists:
        items = info_list.find_all('li')
        for item in items:
            text_line = [line.strip() for line in item.text.strip().split('\n') if line.strip()]

            if len(text_line) >= 2:
                key = text_line[0].replace(':', '')
                value = text_line[-1]

                if key and value: 
                    fighter_data[key] = value
    
    return fighter_data

In [33]:
for name, url in urls.items():
    soup = get_fighter_html(url)
    stats = extract_fighter_stats(soup, name)
    print(f"\n Estatísticas de {name}")

    for k, v in stats.items():
        print(f"{k}: {v}")


 Estatísticas de Charles Oliveira
name: Charles Oliveira
Height: 5' 10"
Weight: 155 lbs.
Reach: 74"
STANCE: Orthodox
DOB: Oct 17, 1989
SLpM: 3.35
Str. Acc.: 54%
SApM: 3.24
Str. Def: 49%
TD Avg.: 2.22
TD Acc.: 40%
TD Def.: 54%
Sub. Avg.: 2.6

 Estatísticas de Max Holloway
name: Max Holloway
Height: 5' 11"
Weight: 155 lbs.
Reach: 69"
STANCE: Orthodox
DOB: Dec 04, 1991
SLpM: 7.20
Str. Acc.: 48%
SApM: 4.74
Str. Def: 59%
TD Avg.: 0.24
TD Acc.: 53%
TD Def.: 83%
Sub. Avg.: 0.3


### Puxando histórico de lutas

In [ ]:
def extract_fight_links_and_basics(soup, fighter_name):
    history = []
    
    table = soup.find('table', class_='b-fight-details__table')
    if not table:
        return history
    
    rows = table.find_all('tr')[1:]
    
    for row in rows:
        cols = row.find_all('td')
        if len(cols) >= 10:
            result_tag = cols[0].find('a')
            if not result_tag:
                continue
                
            result = result_tag.text.strip()
            fight_url = result_tag['href'] 
            
            fighters_tags = cols[1].find_all('a')
            if len(fighters_tags) == 2:
                name1 = ' '.join(fighters_tags[0].text.split())
                name2 = ' '.join(fighters_tags[1].text.split())
                opponent = name1 if name1 != fighter_name else name2
            else:
                opponent = "desconhecido"
                
            fight_data = {
                "Fighter": fighter_name,
                "Opponent": opponent,
                "Result": result,
                "Fight_URL": fight_url, 
            }
            history.append(fight_data)
            
    return history


In [36]:
history_max_v2 = extract_fight_links_and_basics(soup_max, "Max Holloway")
for f in history_max_v2[:2]:
    print(f)

{'Fighter': 'Max Holloway', 'Opponent': 'Dustin Poirier', 'Result': 'win', 'Fight_URL': 'http://ufcstats.com/fight-details/d7507e2741fd4d05'}
{'Fighter': 'Max Holloway', 'Opponent': 'Ilia Topuria', 'Result': 'loss', 'Fight_URL': 'http://ufcstats.com/fight-details/ebf7cea27b83c432'}


In [51]:
def get_fight_totals(fight_url, fighter_name):
    soup = get_fighter_html(fight_url)
    if not soup:
        return {}

    details = {"Fight_URL": fight_url}

    weight_tag = soup.find('i', class_='b-fight-details__fight-title')
    details['Weight_Class'] = weight_tag.text.strip() if weight_tag else 'Unknown'

    tables = soup.find_all('table', class_='b-fight-details__table')

    def get_totals_row(table):
        rows = [r for r in table.find_all('tr') if r.find('td')]
        return rows[-1] if rows else None

    def extract_from_row(row, col_map, fighter_name, prefix=''):
        cols = row.find_all('td')
        names = [p.text.strip() for p in cols[0].find_all('p')]
        if len(names) < 2:
            return {}
        idx = 0 if names[0] == fighter_name else 1
        result = {}
        for c_idx, name in col_map.items():
            ps = cols[c_idx].find_all('p')
            if len(ps) == 2:
                result[f'Fighter_{name}'] = ps[idx].text.strip()
                result[f'Opponent_{name}'] = ps[1-idx].text.strip()
        return result

    col_map_t0 = {1:'KD', 2:'SigStr', 3:'SigStr_pct', 4:'TotalStr', 5:'Td', 6:'Td_pct', 7:'Sub_Att', 9:'Ctrl'}
    col_map_t1 = {1:'SigStr', 3:'Head', 4:'Body', 5:'Leg', 6:'Distance', 7:'Clinch', 8:'Ground'}

    if len(tables) >= 1:
        row0 = get_totals_row(tables[0])
        if row0:
            details.update(extract_from_row(row0, col_map_t0, fighter_name))

    if len(tables) >= 2:
        row1 = get_totals_row(tables[1])
        if row1:
            details.update(extract_from_row(row1, col_map_t1, fighter_name))

    return details


all_fight_data = []

for name, url in urls.items():
    soup = get_fighter_html(url)
    fight_links = extract_fight_links_and_basics(soup, name)
    
    for i, fight in enumerate(fight_links):
        fight_data = get_fight_totals(fight['Fight_URL'], name)
        fight_data['Fighter'] = name
        fight_data['Opponent'] = fight['Opponent']
        fight_data['Result'] = fight['Result']
        all_fight_data.append(fight_data)
        
        time.sleep(0.3)
        if (i + 1) % 5 == 0:
            print(f"  {i + 1} lutas coletadas")

print(f"\nTotal de lutas coletadas: {len(all_fight_data)}")
df_fights_detailed = pd.DataFrame(all_fight_data)
df_fights_detailed.to_parquet('../data/raw/fight_details.parquet', index=False)
print("Salvo em data/raw/fight_details.csv!")
display(df_fights_detailed.head(3))


  5 lutas coletadas
  10 lutas coletadas
  15 lutas coletadas
  20 lutas coletadas
  25 lutas coletadas
  30 lutas coletadas
  35 lutas coletadas
  5 lutas coletadas
  10 lutas coletadas
  15 lutas coletadas
  20 lutas coletadas
  25 lutas coletadas
  30 lutas coletadas

Total de lutas coletadas: 67
Salvo em data/raw/fight_details.csv!


,Fight_URL,Weight_Class,Fighter_KD,Opponent_KD,Fighter_SigStr,Opponent_SigStr,Fighter_SigStr_pct,Opponent_SigStr_pct,Fighter_TotalStr,Opponent_TotalStr,...,Opponent_Leg,Fighter_Distance,Opponent_Distance,Fighter_Clinch,Opponent_Clinch,Fighter_Ground,Opponent_Ground,Fighter,Opponent,Result
0,http://ufcstats.com/fight-details/79865287be90...,Lightweight Bout,0,0,8 of 25,14 of 35,32%,40%,9 of 26,14 of 35,...,0 of 0,7 of 24,14 of 35,1 of 1,0 of 0,0 of 0,0 of 0,Charles Oliveira,Mateusz Gamrot,win
1,http://ufcstats.com/fight-details/7a64d63e1261...,UFC Lightweight Title Bout,0,1,9 of 16,21 of 29,56%,72%,10 of 18,24 of 32,...,1 of 3,8 of 14,18 of 24,1 of 2,0 of 0,0 of 0,3 of 5,Charles Oliveira,Ilia Topuria,loss
2,http://ufcstats.com/fight-details/4f4189009a19...,Lightweight Bout,0,0,12 of 17,40 of 47,70%,85%,16 of 21,52 of 59,...,1 of 1,12 of 17,16 of 23,0 of 0,0 of 0,0 of 0,24 of 24,Charles Oliveira,Michael Chandler,win


### Salvando o parquet

In [25]:
all_stats = []
all_fights = []

for name, url in urls.items():
    soup = get_fighter_html(url)

    if soup:
        fighter_stats = extract_fighter_stats(soup, name)
        all_stats.append(fighter_stats)
        
        fighter_history = extract_fight_history(soup, name)
        all_fights.extend(fighter_history) 

df_stats = pd.DataFrame(all_stats)
df_fights = pd.DataFrame(all_fights)

display(df_stats.head())

import os
os.makedirs("../data/raw", exist_ok=True)
df_stats.to_parquet("../data/raw/fighter_stats.parquet", index=False)
df_fights.to_parquet("../data/raw/fight_history.parquet", index=False)

,name,Height,Weight,Reach,STANCE,DOB,SLpM,Str. Acc.,SApM,Str. Def,TD Avg.,TD Acc.,TD Def.,Sub. Avg.
0,Charles Oliveira,"5' 10""",155 lbs.,"74""",Orthodox,"Oct 17, 1989",3.35,54%,3.24,49%,2.22,40%,54%,2.6
1,Max Holloway,"5' 11""",155 lbs.,"69""",Orthodox,"Dec 04, 1991",7.20,48%,4.74,59%,0.24,53%,83%,0.3
